In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_cricket=pd.DataFrame(pd.read_csv('./datasets/age_group.csv'))

In [ ]:
df_cricket.head()

In [ ]:
df_cricket.describe(include='all')

In [ ]:
def calculate_entropy(series):
    counts = series.value_counts()
    probabilities = counts / len(series)
    entropy = -np.sum(probabilities * np.log2(probabilities))
    return entropy

# Calculate the entropy of the target variable 'Play_cricket'
entropy_play_cricket = calculate_entropy(df_cricket['Play_cricket'])
print(f"Entropy of 'Play_cricket': {entropy_play_cricket:.3f}")

In [ ]:
def calculate_information_gain(df, feature, target):
    total_entropy = calculate_entropy(df[target])

    weighted_entropy = 0
    for value in df[feature].unique():
        subset = df[df[feature] == value]
        subset_entropy = calculate_entropy(subset[target])
        weight = len(subset) / len(df)
        weighted_entropy += weight * subset_entropy

    information_gain = total_entropy - weighted_entropy
    return information_gain

# Calculate Information Gain for 'Age_group'
ig_age_group = calculate_information_gain(df_cricket, 'Age_group', 'Play_cricket')
print(f"Information Gain for 'Age_group': {ig_age_group:.3f}")

# Calculate Information Gain for 'Gender'
ig_gender = calculate_information_gain(df_cricket, 'Gender', 'Play_cricket')
print(f"Information Gain for 'Gender': {ig_gender:.3f}")

# Determine the best split
if ig_age_group > ig_gender:
    best_split_feature = 'Age_group'
    best_ig = ig_age_group
else:
    best_split_feature = 'Gender'
    best_ig = ig_gender

print(f"\nThe best split is based on the feature: {best_split_feature} with Information Gain: {best_ig:.3f}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df_cricket, x=best_split_feature, hue='Play_cricket', palette='viridis')
plt.title(f'Distribution of Play_cricket by {best_split_feature}')
plt.xlabel(best_split_feature)
plt.ylabel('Count')
plt.show()

In [ ]:
# Install graphviz if not already installed (uncomment and run if needed)
# !pip install graphviz

from sklearn.tree import DecisionTreeClassifier, export_graphviz
import graphviz

# Prepare the data for the decision tree.
# We need to convert 'Gender' and 'Play_cricket' into numerical formats.

df_tree = df_cricket.copy()
df_tree['Gender_encoded'] = df_tree['Gender'].map({'M': 0, 'F': 1})
df_tree['Play_cricket_encoded'] = df_tree['Play_cricket'].map({'N': 0, 'Y': 1})

X_tree = df_tree[['Gender_encoded']]
y_tree = df_tree['Play_cricket_encoded']

# Create and train a simple Decision Tree Classifier
tree_classifier = DecisionTreeClassifier(max_depth=1) # We only care about the first split
tree_classifier.fit(X_tree, y_tree)

# Export the trained decision tree to a DOT format
dot_data = export_graphviz(tree_classifier,
                           feature_names=['Gender'],
                           class_names=['N', 'Y'],
                           filled=True, rounded=True,
                           special_characters=True)

# Render the DOT data using graphviz
graph = graphviz.Source(dot_data)
display(graph)

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_graphviz
import graphviz

# The df dataframe is already preprocessed with Label Encoding and missing values filled.
# X and y are already defined from previous steps (Loan_Status is the target, Loan_ID dropped).
# X_train, X_test, y_train, y_test are also ready.

# Let's calculate the entropy of the target variable 'Loan_Status' from the full dataset
entropy_loan_status = calculate_entropy(df_cricket['Loan_Status'])
print(f"Entropy of 'Loan_Status': {entropy_loan_status:.3f}")

In [ ]:
df_loan=pd.DataFrame(pd.read_csv('./datasets/loan.csv'))

In [ ]:
# now using random forest on age_group dataset
from sklearn.ensemble import RandomForestClassifier

# Initialize and train a Random Forest Classifier
# Using n_estimators (number of trees) and a random_state for reproducibility
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train, y_train)

# Make predictions on the test set
y_pred_rf = rf_classifier.predict(X_test)

# Calculate accuracy
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Classifier Accuracy: {accuracy_rf:.3f}")

# Calculate error rate
error_rate_rf = 1 - accuracy_rf
print(f"Random Forest Classifier Error Rate: {error_rate_rf:.3f}")

print("\nClassification Report (Random Forest):")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix (Random Forest):")
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
df_loan.head()

In [ ]:
df_loan=df_loan.drop("Loan_ID", axis=1)
df_loan.head()

In [ ]:
def calculate_entropy(series):
    counts = series.value_counts()
    probabilities = counts / len(series)
    entropy = -np.sum(probabilities * np.log2(probabilities))
    return entropy

In [ ]:
# entrophy of whole dataset
entropy_loan_status = calculate_entropy(df_loan['Loan_Status'])
print(f"Entropy of 'Loan_Status': {entropy_loan_status:.3f}")

In [ ]:
# calculating the information gain for each feature in the loan dataset
def calculate_information_gain(df, feature, target):
    total_entropy = calculate_entropy(df[target])

    weighted_entropy = 0
    for value in df[feature].unique():
        subset = df[df[feature] == value]
        subset_entropy = calculate_entropy(subset[target])
        weight = len(subset) / len(df)
        weighted_entropy += weight * subset_entropy

    information_gain = total_entropy - weighted_entropy
    return information_gain

# passing the features and target variable to the function
features = df_loan.columns[:-1]  # Exclude the target variable 'Loan_Status
for feature in features:
    ig = calculate_information_gain(df_loan, feature, 'Loan_Status')
    print(f"Information Gain for '{feature}': {ig:.3f}")


In [ ]:
# now we will determine the best split based on the highest information gain
best_split_feature = None
best_split_feature_index = None
best_ig = -1
best_ig_index = -1
for i, feature in enumerate(features):
    ig = calculate_information_gain(df_loan, feature, 'Loan_Status')
    if ig > best_ig:
        best_ig = ig
        best_split_feature = feature
        best_ig_index = i
print(f"\nThe best split is based on the feature: {best_split_feature} with Information Gain: {best_ig:.3f}")


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
target = "Loan_Status"
X = df_loan.drop([target, 'Loan_ID'], axis=1)
y = df_loan[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and train a Random Forest Classifier
# Using n_estimators (number of trees) and a random_state for reproducibility
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train, y_train)

# Make predictions on the test set
y_pred_rf = rf_classifier.predict(X_test)

# Calculate accuracy
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Classifier Accuracy: {accuracy_rf:.3f}")

# Calculate error rate
error_rate_rf = 1 - accuracy_rf
print(f"Random Forest Classifier Error Rate: {error_rate_rf:.3f}")

print("\nClassification Report (Random Forest):")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix (Random Forest):")
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_graphviz
import graphviz

# The df dataframe is already preprocessed with Label Encoding and missing values filled.
# X and y are already defined from previous steps (Loan_Status is the target, Loan_ID dropped).
# X_train, X_test, y_train, y_test are also ready.

# Let's calculate the entropy of the target variable 'Loan_Status' from the full dataset
entropy_loan_status = calculate_entropy(df_loan['Loan_Status'])
print(f"Entropy of 'Loan_Status': {entropy_loan_status:.3f}")

In [ ]:
potential_split_features = ['Gender', 'Married', 'Education', 'Self_Employed', 'Credit_History', 'Property_Area']
information_gains = {}

for feature in potential_split_features:
    # Ensure the feature is in the DataFrame and properly encoded
    if feature in df_loan.columns:
        ig = calculate_information_gain(df_loan, feature, 'Loan_Status')
        information_gains[feature] = ig
        print(f"Information Gain for '{feature}': {ig:.3f}")

# Find the feature with the highest information gain
best_split_feature_loan = max(information_gains, key=information_gains.get)
best_ig_loan = information_gains[best_split_feature_loan]

print(f"\nThe best split for Loan Prediction is based on the feature: {best_split_feature_loan} with Information Gain: {best_ig_loan:.3f}")

In [ ]:
# For visualization, let's focus on the feature identified as the best split (or a strong one like Credit_History)
# We'll use the pre-encoded X and y from the previous steps.

tree_classifier_loan = DecisionTreeClassifier(max_depth=3, random_state=42) # Limiting depth for clarity
tree_classifier_loan.fit(X_train, y_train)

# Feature names for visualization (using X_train columns as they are already encoded)
feature_names_loan = X_train.columns.tolist()

# Class names for the target variable 'Loan_Status'
class_names_loan = ['N', 'Y'] # Assuming 0 is 'N' (Rejected) and 1 is 'Y' (Approved) after Label Encoding

# Export the trained decision tree to a DOT format
dot_data_loan = export_graphviz(tree_classifier_loan,
                                feature_names=feature_names_loan,
                                class_names=class_names_loan,
                                filled=True, rounded=True,
                                special_characters=True,
                                impurity=False) # Set impurity to False for cleaner look if preferred

# Render the DOT data using graphviz
graph_loan = graphviz.Source(dot_data_loan)
display(graph_loan)
